# Configs

In [1]:
import pandas as pd
import numpy as np

In [2]:
Proj_path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/FG_2026_Projections'

In [3]:
Player_ID_Map_Path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/Player_ID_Map/Latest_Player_ID_Map.csv'

In [4]:
Proj_date = 'Jan23_2026'

In [5]:
Projections_dates = {
    'ATC':'Jan23_2026',
    'OOPSY':'Jan23_2026'
}

In [6]:
Proj_weights = {
    'ATC':.4,
    'OOPSY':.6
}

In [7]:
Hitting_Possitions = [
    'C',
    '1B',
    '2B',
    'SS',
    '3B',
    'OF',
    'DH'
]
Pitching_Possitions = [
    'SP',
    'RP'
]

In [8]:
Player_id_cols = [
    'FG ID', 'Team'#,'MLBAMID','Name','Team','NameASCII'
]
Hitter_Count_Stats = [
    'G','PA','AB','H','1B','2B','3B','HR','R','RBI','BB','HBP','SF','WAR','ADP'
]
Pitcher_Count_Stats = [
    'W', 'L', 'QS', 'G', 'GS', 'SV', 'HLD', 'IP', 'TBF', 'H', 'R', 'ER', 'HR', 'BB', 'HBP', 'SO','WAR','ADP'
] # 'BS',

In [9]:
def w_avg(df, values, weights):
    d = df[values]
    w = df[weights]
    return (d * w).sum() / w.sum()

In [10]:
Hitter_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Hitting_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        temp_df.rename(columns={
            'PlayerId':'FG ID'
        },inplace=True)
        temp_df['FG ID'] = temp_df['FG ID'].astype(str)
        temp_df['FG ID'] = temp_df['FG ID'].str.replace('.0','')
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Hitter_Count_Stats)['POS'].agg(list).reset_index()
        )
    Hitter_Projections_df = pd.concat([Hitter_Projections_df,this_proj_df], ignore_index=True)
Hitter_Projections_df = Hitter_Projections_df[Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Hitter_Projections_df.groupby(Player_id_cols+Hitter_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )


Pitcher_Projections_df = pd.DataFrame()
for proj,date in Projections_dates.items():
    this_proj_df = pd.DataFrame()
    for pos in Pitching_Possitions:
        temp_df = pd.read_csv(f"{Proj_path}/{date}/{proj}/fangraphs-{proj}_{date}-projections_{pos}.csv")
        temp_df['Projection'] = proj
        temp_df['Proj_weight'] = Proj_weights[proj]
        temp_df['POS'] = pos
        temp_df.rename(columns={
            'PlayerId':'FG ID'
        },inplace=True)
        temp_df['FG ID'] = temp_df['FG ID'].astype(str)
        temp_df['FG ID'] = temp_df['FG ID'].str.replace('.0','')
        this_proj_df = pd.concat([this_proj_df, temp_df], ignore_index=True)
    this_proj_df = this_proj_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                this_proj_df.groupby(Player_id_cols+Pitcher_Count_Stats)['POS'].agg(list).reset_index()
        )
    Pitcher_Projections_df = pd.concat([Pitcher_Projections_df,this_proj_df], ignore_index=True)
Pitcher_Projections_df = Pitcher_Projections_df[Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight']].drop_duplicates().merge(
                Pitcher_Projections_df.groupby(Player_id_cols+Pitcher_Count_Stats+['Projection','Proj_weight'])['POS'].agg(list).reset_index()
        )

In [11]:
for col in Player_id_cols:
    Hitter_Projections_df[col] = Hitter_Projections_df[col].astype(str)
    Pitcher_Projections_df[col] = Pitcher_Projections_df[col].astype(str)

In [12]:
Player_ID_Map_df = pd.read_csv(Player_ID_Map_Path)
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].astype(str)
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].fillna(Player_ID_Map_df['FG Minor ID'])
Player_ID_Map_df['FG ID'] = Player_ID_Map_df['FG ID'].str.replace('.0','')
Player_ID_Map_df['Ottoneu ID'] = Player_ID_Map_df['Ottoneu ID'].astype(str)
Player_ID_Map_df['Ottoneu ID'] = Player_ID_Map_df['Ottoneu ID'].str.replace('.0','')

In [13]:
Pitcher_Projections_df.head()

,FG ID,Team,W,L,QS,G,GS,SV,HLD,IP,...,ER,HR,BB,HBP,SO,WAR,ADP,Projection,Proj_weight,POS
0,22267,DET,14.3742,7.04161,18.1585,29.8766,29.8766,0.0,0.000000,189.747,...,58.4961,18.7818,38.9320,6.39001,229.190,5.84816,5.840000,ATC,0.4,[[SP]]
1,33677,PIT,12.3671,8.20263,17.6124,30.2106,30.2106,0.0,0.000000,186.614,...,57.2135,15.5018,45.5249,6.72205,224.303,5.78243,9.610000,ATC,0.4,[[SP]]
2,27463,BOS,13.2469,8.14561,15.5400,29.3089,29.3089,0.0,0.000000,180.506,...,61.6010,19.3156,46.3108,5.27536,221.171,5.01687,10.100000,ATC,0.4,[[SP]]
3,17995,SFG,12.9634,10.14590,17.1698,30.3029,30.3029,0.0,0.187813,195.268,...,72.5313,15.7501,44.8790,5.85560,183.708,4.80906,56.419998,ATC,0.4,[[SP]]
4,20778,PHI,12.3766,7.77394,16.5268,29.3337,29.3337,0.0,0.000000,185.717,...,65.7470,16.0440,46.2015,5.82465,179.174,4.46801,27.580000,ATC,0.4,[[SP]]


In [14]:
Pitcher_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Pitcher_Count_Stats], weights=x["Proj_weight"], axis=0), Pitcher_Count_Stats))

,,W,L,QS,G,GS,SV,HLD,IP,TBF,H,R,ER,HR,BB,HBP,SO,WAR,ADP
FG ID,Team,,,,,,,,,,,,,,,,,,
10061,NYM,3.005452,2.556932,0.00000,60.69940,0.00000,0.652374,13.962000,59.70140,253.81240,50.97252,25.918360,23.89292,7.450292,22.096120,3.064604,61.41672,0.354945,749.419983
10078,CHC,2.945732,2.938668,0.00000,65.70708,0.00000,1.035276,15.881800,64.51868,268.56040,54.63908,28.738160,26.03112,7.948096,22.533800,2.474020,66.48364,0.399697,748.940002
10233,BOS,3.355612,3.199640,0.00000,61.23632,0.00000,30.385000,2.353536,60.55644,248.87200,43.44752,21.783240,19.89656,5.717836,26.861360,2.596076,82.58592,1.232904,46.840000
10310,PHI,9.304912,5.024288,12.69184,20.43608,20.43608,0.000000,0.000000,123.62880,500.63440,99.19724,47.291120,43.36380,14.797160,33.475240,5.116168,144.48120,2.875334,148.449997
10315,MIN,1.675228,1.610888,0.00000,31.58100,0.00000,0.012948,0.359376,32.26188,138.90000,31.07328,16.849120,15.76520,4.588596,11.817920,2.036932,29.46212,0.024503,999.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sa917950,NYY,0.000000,0.000000,0.00000,1.00000,0.00000,0.000000,0.000000,1.55146,7.00000,2.00000,1.000000,1.00000,0.000000,1.000000,0.000000,1.00000,-0.002742,999.000000
sa918182,CIN,0.000000,0.000000,0.00000,1.00000,1.00000,0.000000,0.000000,5.33333,20.00000,4.00000,3.000000,3.00000,1.000000,2.000000,0.000000,5.00000,0.079048,999.000000
sa920454,PHI,0.000000,0.000000,0.00000,1.00000,0.00000,0.000000,0.000000,1.18531,5.00000,1.00000,1.000000,1.00000,0.000000,1.000000,0.000000,1.00000,-0.003729,999.000000


In [15]:
Hitter_weighted_averages = Hitter_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Hitter_Count_Stats], weights=x["Proj_weight"], axis=0), Hitter_Count_Stats)).reset_index()
Pitcher_weighted_averages = Pitcher_Projections_df.groupby(Player_id_cols).apply(lambda x: pd.Series(np.average(x[Pitcher_Count_Stats], weights=x["Proj_weight"], axis=0), Pitcher_Count_Stats)).reset_index()

In [16]:
Hitter_weighted_averages.shape[0]

1981

In [17]:
Pitcher_weighted_averages.shape[0]

3041

In [18]:
Hitter_Projections_df['POS'] = Hitter_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])
Pitcher_Projections_df['POS'] = Pitcher_Projections_df['POS'].apply(lambda pos_list_of_lists: pos_list_of_lists[0])

In [19]:
Hitter_pos_merged = Hitter_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()
Pitcher_pos_merged = Pitcher_Projections_df.groupby(Player_id_cols)['POS'].apply(lambda pos_list: list(set(sum(pos_list,[])))).reset_index()

In [20]:
Hitter_pos_merged.shape[0]

1981

In [21]:
Pitcher_pos_merged.shape[0]

3041

In [22]:
Hitter_final_merge_df=Hitter_weighted_averages.merge(Hitter_pos_merged,on=Player_id_cols,how='outer')
Pitcher_final_merge_df=Pitcher_weighted_averages.merge(Pitcher_pos_merged,on=Player_id_cols,how='outer')

In [23]:
Hitter_final_merge_df = Hitter_final_merge_df.merge(Player_ID_Map_df[['FG ID','Ottoneu ID','Ottoneu Positions','Name']],how='inner')
Pitcher_final_merge_df = Pitcher_final_merge_df.merge(Player_ID_Map_df[['FG ID','Ottoneu ID','Ottoneu Positions','Name']],how='inner')

In [24]:
# Find Duplicate Players
Hitter_final_merge_df[Hitter_final_merge_df['Ottoneu ID'].isin([key for key, val in Hitter_final_merge_df['Ottoneu ID'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name


In [25]:
Pitcher_final_merge_df[Pitcher_final_merge_df['Ottoneu ID'].isin([key for key, val in Pitcher_final_merge_df['Ottoneu ID'].value_counts().to_dict().items() if val > 1])].sort_values('Name')

,FG ID,Team,W,L,QS,G,GS,SV,HLD,IP,...,HR,BB,HBP,SO,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name


In [26]:
Hitter_final_merge_df.head()

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name
0,10028,CHC,0.0000,1.0000,1.0000,0.0000,0.00000,0.00000,0.000000,0.00000,...,0.00000,0.00000,0.000000,0.000000,0.000169,999.000000,[C],18048,C,Christian Bethancourt
1,10155,LAA,129.1444,556.3820,472.4704,113.9432,65.56196,19.01020,1.917732,27.45316,...,73.83592,73.21920,6.691312,3.828264,2.320814,214.580002,[OF],6305,OF,Mike Trout
2,10642,HOU,0.0000,1.0000,1.0000,0.0000,0.00000,0.00000,0.000000,0.00000,...,0.00000,0.00000,0.000000,0.000000,0.001724,999.000000,[C],9791,C,Carlos Perez
3,10815,ATL,144.9268,627.0208,543.6880,134.1240,90.24152,26.13876,0.992002,16.75172,...,67.43156,69.00872,9.141040,4.581340,1.197811,183.520004,[OF],13702,OF,Jurickson Profar
4,11477,MIL,142.5496,612.5212,536.5336,137.0560,92.19288,23.32544,1.235164,19.70256,...,71.73596,66.64432,4.782532,3.651592,1.773006,140.289993,[OF],14893,OF,Christian Yelich


In [27]:
# Fix known Pitcher Hitter overlaps, manually choosing which position for these players.
# Note: This explicitly leaves some players as duplicates because they are 2-way players. (e.g. Otahni)

PlayerIds_drop_Pitching = []
PlayerIds_drop_Hitting = []

# Andrés Nolaya sa3018563
PlayerIds_drop_Pitching.append('sa3018563')

# Ben Malgeri sa3016975
PlayerIds_drop_Pitching.append('sa3016975')

# Caleb Pendleton sa3022483
PlayerIds_drop_Pitching.append('sa3022483')

# Carlos Franco sa3015580
PlayerIds_drop_Pitching.append('sa3015580')

# Chase Valentine sa3020002
PlayerIds_drop_Pitching.append('sa3020002')

# Curtis Washington Jr. sa3019803	
PlayerIds_drop_Pitching.append('sa3019803')

# David Leon sa3016336
PlayerIds_drop_Pitching.append('sa3016336')

# Diego Viloria sa3016299
PlayerIds_drop_Hitting.append('sa3016299')

# Fernando Caldera sa3015547
PlayerIds_drop_Pitching.append('sa3015547')

# Jackson Cluff sa3010342
PlayerIds_drop_Hitting.append('sa3010342')

# Jesus Castro sa3019090
PlayerIds_drop_Hitting.append('sa3019090')

# Johan Contreras sa3019217
PlayerIds_drop_Pitching.append('sa3019217')

# Jose Gerardo sa3018645
PlayerIds_drop_Hitting.append('sa3018645')

# Keduar Trujillo sa3019764
PlayerIds_drop_Pitching.append('sa3019764')

# Sean Barnett sa3025496
PlayerIds_drop_Hitting.append('sa3025496')

# Wyatt Hoffman sa3018219
PlayerIds_drop_Pitching.append('sa3018219')

# Yolmer Sánchez 11602
PlayerIds_drop_Pitching.append('11602')

# Anthony Prato	sa1052903
PlayerIds_drop_Pitching.append('sa1052903')

# Nick Kahle sa1115979
PlayerIds_drop_Pitching.append('sa1115979')

# Dylan Shockley sa1170874
PlayerIds_drop_Pitching.append('sa1170874')

# Andy Weber sa3008282
PlayerIds_drop_Pitching.append('sa3008282')

# Maikol Escotto sa3008878
PlayerIds_drop_Pitching.append('sa3008878')

# Myles Emmerson sa3016862
PlayerIds_drop_Pitching.append('sa3016862')

# Cole Urman sa3023270
PlayerIds_drop_Pitching.append('sa3023270')

In [28]:
Pitcher_final_merge_df = Pitcher_final_merge_df[~Pitcher_final_merge_df['FG ID'].isin(PlayerIds_drop_Pitching)]
Hitter_final_merge_df = Hitter_final_merge_df[~Hitter_final_merge_df['FG ID'].isin(PlayerIds_drop_Hitting)]

In [29]:
Hitter_final_merge_df.head()

,FG ID,Team,G,PA,AB,H,1B,2B,3B,HR,...,RBI,BB,HBP,SF,WAR,ADP,POS,Ottoneu ID,Ottoneu Positions,Name
0,10028,CHC,0.0000,1.0000,1.0000,0.0000,0.00000,0.00000,0.000000,0.00000,...,0.00000,0.00000,0.000000,0.000000,0.000169,999.000000,[C],18048,C,Christian Bethancourt
1,10155,LAA,129.1444,556.3820,472.4704,113.9432,65.56196,19.01020,1.917732,27.45316,...,73.83592,73.21920,6.691312,3.828264,2.320814,214.580002,[OF],6305,OF,Mike Trout
2,10642,HOU,0.0000,1.0000,1.0000,0.0000,0.00000,0.00000,0.000000,0.00000,...,0.00000,0.00000,0.000000,0.000000,0.001724,999.000000,[C],9791,C,Carlos Perez
3,10815,ATL,144.9268,627.0208,543.6880,134.1240,90.24152,26.13876,0.992002,16.75172,...,67.43156,69.00872,9.141040,4.581340,1.197811,183.520004,[OF],13702,OF,Jurickson Profar
4,11477,MIL,142.5496,612.5212,536.5336,137.0560,92.19288,23.32544,1.235164,19.70256,...,71.73596,66.64432,4.782532,3.651592,1.773006,140.289993,[OF],14893,OF,Christian Yelich


In [30]:
#Test Hitter / Pitcher Concat
Quick_test = pd.concat([Hitter_final_merge_df,Pitcher_final_merge_df])
Quick_test[Quick_test['Ottoneu ID'].isin([key for key, val in Quick_test['Ottoneu ID'].value_counts().to_dict().items() if val > 1])][['Name','Ottoneu ID','FG ID','PA','IP']]

,Name,Ottoneu ID,FG ID,PA,IP
168,Shohei Ohtani,33600,19755,675.76,NaN
238,Shohei Ohtani,33600,19755,NaN,111.9524


In [31]:
Hitter_final_merge_df.to_csv(Proj_path+f'/my_Hitter_Proj_{Proj_date}.csv',index=False)

In [32]:
Pitcher_final_merge_df.to_csv(Proj_path+f'/my_Pitcher_Proj_{Proj_date}.csv',index=False)